**⭐ 1. What This Pattern Solves**

Handles data where a single logical record spans multiple lines (common in logs, CSVs with multiline fields, or stack traces).
Reconstructs full records before downstream ETL or transformation.
Prevents partial or broken data from entering analytics pipelines.
Essential for batch processing or streaming ingestion where line breaks do not correspond to record boundaries.

**⭐ 2. SQL Equivalent**

In [0]:
%sql
-- Conceptual equivalent using window functions or string aggregation
WITH raw_lines AS (
    SELECT *,
           SUM(CASE WHEN is_start = 1 THEN 1 ELSE 0 END) OVER (ORDER BY line_id) AS record_id
    FROM logs
)
SELECT record_id,
       STRING_AGG(line_text, '\n') AS full_record
FROM raw_lines
GROUP BY record_id;

**⭐ 3. Core Idea**

Accumulate lines until a clear record boundary is detected, then emit a single reconstructed record.

**⭐ 4. Template Code (MEMORIZE THIS)**

In [0]:
reconstructed = []
buffer = []

for line in input_lines:
    if is_start_of_record(line):
        if buffer:
            reconstructed.append("\n".join(buffer))
            buffer = []
    buffer.append(line)

if buffer:
    reconstructed.append("\n".join(buffer))

# reconstructed now contains full records

**⭐ 5. Detailed Example**

In [0]:
2025-12-16 INFO Start process
Details: step1
Details: step2
2025-12-16 INFO End process

input_lines = [
    "2025-12-16 INFO Start process",
    "Details: step1",
    "Details: step2",
    "2025-12-16 INFO End process"
]

def is_start_of_record(line):
    return line.startswith("2025-12-16 INFO")  # simple boundary detection

reconstructed = []
buffer = []

for line in input_lines:
    if is_start_of_record(line):
        if buffer:
            reconstructed.append("\n".join(buffer))
            buffer = []
    buffer.append(line)

if buffer:
    reconstructed.append("\n".join(buffer))

for rec in reconstructed:
    print("---")
    print(rec)

---
2025-12-16 INFO Start process
Details: step1
Details: step2
---
2025-12-16 INFO End process

**⭐ 6. Mini Practice Problems**

Reconstruct multiline JSON logs where a record starts with { and ends with }.
Merge stack traces from an error log where each trace starts with Exception:.
Combine CSV rows that are split due to embedded newlines inside quoted fields.

**⭐ 7. Full Data Engineering Scenario**

Problem Statement:
You ingest server logs where each request spans multiple lines. Lines starting with a timestamp indicate a new request. Downstream analytics need each request as a single record.

Expected Output:
A list where each element contains the full request log (multi-line).

Skeleton Solution:

In [0]:
def reconstruct_logs(lines):
    records = []
    buffer = []
    for line in lines:
        if line_starts_with_timestamp(line):
            if buffer:
                records.append("\n".join(buffer))
                buffer = []
        buffer.append(line)
    if buffer:
        records.append("\n".join(buffer))
    return records

**⭐ 8. Time & Space Complexity**

Time Complexity: O(n), n = total number of lines (single pass).
Space Complexity: O(m), m = total number of characters in all reconstructed records.

**⭐ 9. Common Pitfalls & Mistakes**

❌ Appending lines before checking for record boundary → duplicates first line in each record.
❌ Using string concatenation (+=) in a loop → quadratic time for large datasets.
❌ Forgetting to flush the buffer at the end → last record lost.
✔ Correct approach: use a list buffer + join, flush on boundary and at end.
✔ Use clear, deterministic record boundary detection (timestamps, markers, or regex).